# Data

In [353]:
import nltk
from nltk.corpus import gutenberg, stopwords
from nltk import tokenize
import autocorrect
from nltk.stem import WordNetLemmatizer
from autocorrect import Speller
nltk.download('omw-1.4')
import regex
from gensim.models import Word2Vec
import gensim.downloader as api
import pandas as pd


[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Hunter\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Instructions
Objective:
To train word embeddings on famous works of Shakespeare and evaluate their understanding.

Data:
The entire text of plays: 1) The Tragedy of Hamlet, Prince of Denmark, 2) The Tragedy of Macbeth, and 3) The Tragedy of Julius Caesar. These are available from the Gutenberg corpus of the NLTK library. Characters and synopses can be found on Wikipedia.

Problem Statement:
Natural language processing is an important part of the most advanced artificial intelligence software we have today. By studying volumes of text, word embeddings are able to elicit meaning from the words within training data. Your goal is to train a word embedding on three famous works of Shakespeare to determine how well your embedding can understand the meaning of character names and other Shakespearean English words found in these plays.

Steps to be completed:
Create a Jupyter notebook and complete the following steps:



# Data
Use nltk.corpus.gutenberg.raw to load the three plays listed above into a single variable and lower the case.


In [354]:
# Assigning the plays to variables
nltk.download('gutenberg')
hamlet = gutenberg.raw('shakespeare-hamlet.txt')
macbeth = gutenberg.raw('shakespeare-macbeth.txt')
julius_caesar = gutenberg.raw('shakespeare-caesar.txt')

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\Hunter\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!



Tokenize the text into sentences, and then each sentence into words.


In [355]:
# Keeping all the text in a single variable with normalized case
lower_text = (hamlet + macbeth + julius_caesar).lower()

In [356]:
# Tokenizing sentences
sentences = tokenize.sent_tokenize(lower_text)

In [357]:
# Tokenizing words
words = [tokenize.word_tokenize(sent) for sent in sentences]

Use Speller from the autocorrect library to correct spelling mistakes. 


In [358]:
# Correcting spelling errors
spell = Speller(lang = 'en')
correct_sentences = [[spell(word) for word in sentence] for sentence in words]

Create a list of stopwords (using publicly available lists and/or adding your own) and remove these.


In [359]:
# Removing stop words
stop_words = set(stopwords.words('english'))
no_stop_words = [[word for word in sentence if word not in stop_words] for sentence in correct_sentences]

Use regular expressions (the re library) to do any additional cleanup of the text you wish to do.


Use PorterStemmer or WordNetLemmatizer from nltk.stem on the text.


In [360]:
# Lemmatizing
lemmatizer = WordNetLemmatizer()
lemmatized_words = [[lemmatizer.lemmatize(word) for word in sentence] for sentence in no_stop_words]

In [361]:
# Removing punctuation and then the empty tokens created by the cleanup
regex_cleaned = [[regex.sub(r'[^a-zA-Z0-9]', '', word) for word in sentence] for sentence in lemmatized_words]
regex_cleaned = [[word for word in sentence if word.strip() != ''] for sentence in regex_cleaned]

Print out the words in the first five sentences of the processed text data. (Viewing this may give you additional ideas for the previous steps.)


In [362]:
# Printing the first five sentences
for i in range(5):
    print(f"Sentence {i}: {regex_cleaned[i]}")

Sentence 0: ['tragedy', 'hamlet', 'william', 'shakespeare', '1599', 'act', 'prime']
Sentence 1: ['scene', 'prima']
Sentence 2: ['enter', 'bernard', 'francisco', 'two', 'sentinel']
Sentence 3: ['bernard']
Sentence 4: ['s']


In [363]:
# Comparing with the actual few sentences
for i in range(5):
    print(f"Sentence {i}: {words[i]}")

Sentence 0: ['[', 'the', 'tragedie', 'of', 'hamlet', 'by', 'william', 'shakespeare', '1599', ']', 'actus', 'primus', '.']
Sentence 1: ['scoena', 'prima', '.']
Sentence 2: ['enter', 'barnardo', 'and', 'francisco', 'two', 'centinels', '.']
Sentence 3: ['barnardo', '.']
Sentence 4: ['who', "'s", 'there', '?']



# Modeling 
Create a CBOW word2vec model from gensim.model. Make choices of vector_size, epochs, window, min_count, and possibly other hyperparameters. Train it on the cleaned Shakespeare text data. Use gensim.model.wv.key_to_index  and gensim.model.wv.get_vecattr to print out a list of the 20 most frequent words in the vocabulary along with the word count. Consider improving the text cleaning steps above based on this information. 


# Processing

## Word2Vec models

### CBOW: target word from context words

In [364]:
model = Word2Vec(sentences=regex_cleaned, vector_size=100, window=5, min_count=3, workers=4, sg=0, epochs=20)

In [365]:
top20 = list(model.wv.key_to_index.keys())[:20]

print("Top 20 most frequent words and their counts:\n")
for word in top20:
    count = model.wv.get_vecattr(word, "count")
    print(f"{word:15}  {count}")

Top 20 most frequent words and their counts:

d                585
ham              337
thou             307
lord             306
shall            300
s                295
come             284
king             248
enter            230
good             221
let              220
mac              205
thy              202
like             200
cesar            193
one              188
make             185
know             184
v                175
thee             174


Based on this word count, we notice a few things:
- d is the top word, which possibly came from words I'd, you'd, we'd. It is just a contraction of 'would', and should probably be removed. Similarly, s and v perhaps correspond to 'is' and 'have'.
- Archaic pronouns like 'thee', 'thy', 'thou' constitute stop words and should also be removed.
- 'Caesar', 'Hamlet' and 'Macbeth' being correct to 'cesar', 'mac', 'ham' sounds lowkey appetizing and defeats the purpose. However, we cannot remove the spelling correction step because the text consists of Latin words which should be converted to English. A better decision might be to manually code those words to preserve the original meaning.

In [366]:
archaic = {"thou", "thee", "thy", "thine", "ye", "hath", "doth"}
stop_words.update(archaic)

In [367]:
regex_cleaned = [[w for w in sent if len(w) > 1] for sent in regex_cleaned]

In [368]:
name_map = {
    "hamlet": "hamlet",
    "ham": "hamlet",
    "macbeth": "macbeth",
    "mac": "macbeth",
    "caesar": "caesar",
    "cesar": "caesar"
}

normalized = [[name_map.get(w, w) for w in sent] for sent in regex_cleaned]

In [369]:
for i in range(5):
    print(f"Sentence {i}: {normalized[i]}")

Sentence 0: ['tragedy', 'hamlet', 'william', 'shakespeare', '1599', 'act', 'prime']
Sentence 1: ['scene', 'prima']
Sentence 2: ['enter', 'bernard', 'francisco', 'two', 'sentinel']
Sentence 3: ['bernard']
Sentence 4: []


In [370]:
model = Word2Vec(sentences=normalized, vector_size=100, window=5, min_count=3, workers=4, sg=0, epochs=20)

In [371]:
top20 = list(model.wv.key_to_index.keys())[:20]

print("Top 20 most frequent words and their counts:\n")
for word in top20:
    count = model.wv.get_vecattr(word, "count")
    print(f"{word:15}  {count}")

Top 20 most frequent words and their counts:

hamlet           444
thou             307
lord             306
shall            300
come             284
macbeth          271
king             248
enter            230
good             221
let              220
thy              202
like             200
caesar           193
one              188
make             185
know             184
thee             174
self             166
would            163
aboutus          162


In [372]:
top_5 = ['hamlet', 'cauldron', 'nature', 'spirit', 'general', 'prythee']
for i in top_5:
    print(f'Most similar results for: {i}')
    print(model.wv.most_similar(i, topn=5))


Most similar results for: hamlet
[('good', 0.9929336309432983), ('polo', 0.99224454164505), ('sweet', 0.9905007481575012), ('faith', 0.9880846738815308), ('king', 0.9879695177078247)]
Most similar results for: cauldron
[('burned', 0.9985411167144775), ('horse', 0.9984645843505859), ('thereto', 0.9984216094017029), ('opinion', 0.9984073042869568), ('side', 0.9983751177787781)]
Most similar results for: nature
[('though', 0.9974195957183838), ('act', 0.9973759055137634), ('even', 0.9972039461135864), ('judgement', 0.9971790313720703), ('whose', 0.9971538186073303)]
Most similar results for: spirit
[('wound', 0.9982137680053711), ('hide', 0.9979503154754639), ('lie', 0.9978554844856262), ('bone', 0.9978185296058655), ('vse', 0.9978169798851013)]
Most similar results for: general
[('common', 0.9988754391670227), ('danger', 0.9988331198692322), ('seek', 0.9988242387771606), ('draw', 0.9987999200820923), ('vile', 0.9987945556640625)]
Most similar results for: prythee
[('farewell', 0.99805736

Create a skipgram word2vec model from gensim.model. Make choices of vector_size, epochs, window, min_count, and possibly other hyperparameters. Train it on the cleaned Shakespeare text data.


In [373]:
model_sg = Word2Vec(sentences=normalized, vector_size=100, window=5, min_count=3, workers=4, sg=1, epochs=20)

In [374]:
top20_sg = list(model_sg.wv.key_to_index.keys())[:20]

print("Top 20 most frequent words and their counts:\n")
for word in top20_sg:
    count = model_sg.wv.get_vecattr(word, "count")
    print(f"{word:15}  {count}")

Top 20 most frequent words and their counts:

hamlet           444
thou             307
lord             306
shall            300
come             284
macbeth          271
king             248
enter            230
good             221
let              220
thy              202
like             200
caesar           193
one              188
make             185
know             184
thee             174
self             166
would            163
aboutus          162


In [375]:
top_5 = ['hamlet', 'thou', 'lord', 'shall', 'come']
for i in top_5:
    print(f'Most similar results for: {i}')
    print(model_sg.wv.most_similar(i, topn=5))

Most similar results for: hamlet
[('polo', 0.9118939638137817), ('lord', 0.8542149066925049), ('sent', 0.8497737646102905), ('were', 0.8472520112991333), ('faith', 0.8406642079353333)]
Most similar results for: thou
[('didst', 0.7782082557678223), ('art', 0.7700484395027161), ('shalt', 0.7621564865112305), ('wouldst', 0.7581943273544312), ('stratum', 0.7455163598060608)]
Most similar results for: lord
[('polo', 0.8551681637763977), ('hamlet', 0.8542150259017944), ('faith', 0.7828043103218079), ('good', 0.7738195657730103), ('colony', 0.7606551051139832)]
Most similar results for: shall
[('elder', 0.8077791333198547), ('please', 0.8028163909912109), ('competes', 0.7988343834877014), ('receive', 0.7972277998924255), ('paint', 0.7859323024749756)]
Most similar results for: come
[('fetch', 0.836710512638092), ('away', 0.831356942653656), ('sit', 0.8156183958053589), ('toot', 0.787418782711029), ('dunsinane', 0.7860831022262573)]


Load the pretrained GloVe model from gensim.models.keyedvectors for comparison with the models trained on Shakespeare text. Use markdown to make note of the data that GloVe has been trained on.

GloVe Training Data
GloVe models available via gensim.downloader were trained on massive general-domain corpora like Wikipedia 2014 + Gigaword 5 (6B tokens, 400K vocab for the 100d version).

In [377]:
# Recommended: glove-wiki-gigaword-100 (matches your vector_size=100)
glove_model = api.load("glove-wiki-gigaword-100")


In [378]:

# Now compare, e.g., on "oeuvre"
print("Skip-gram (Shakespeare):", model_sg.wv.most_similar("hamlet", topn=5))
print("GloVe:", glove_model.most_similar("hamlet", topn=5))

Skip-gram (Shakespeare): [('polo', 0.9118939638137817), ('lord', 0.8542149066925049), ('sent', 0.8497737646102905), ('were', 0.8472520112991333), ('faith', 0.8406642079353333)]
GloVe: [('village', 0.6998987197875977), ('town', 0.6558532118797302), ('situated', 0.5926076769828796), ('located', 0.5660547614097595), ('unincorporated', 0.5599358677864075)]


# Discussion
Compare the three models by finding the 5 most similar terms to each of the following terms: 'hamlet', 'cauldron', 'nature', 'spirit', 'general', and 'prythee'. Comment on how well each model captured the meaning of the word, and if there are multiple meanings, which meaning was given.


In [379]:
terms = ['hamlet', 'cauldron', 'nature', 'spirit', 'general', 'prythee']

models = [model, model_sg, glove_model]

# Dictionary to collect ALL results
results_dict = {
    'CBOW': {},
    'Skip-gram': {},
    'GloVe': {}
}

print("Object IDs:")
print(f"model id: {id(model)}")
print(f"model_sg id: {id(model_sg)}")
print(f"glove_model id: {id(glove_model)}")
print(f"glove_model type: {type(glove_model)}")

# CBOW
print("=== CBOW ===")
for i in terms:
    try:
        top5 = model.wv.most_similar(i, topn=5)
        print(f'{i}: {top5}')
        results_dict['CBOW'][i] = top5  # Store list of tuples
    except KeyError:
        print(f'{i}: KeyError')
        results_dict['CBOW'][i] = 'KeyError'

# Skip-gram  
print("\n=== Skip-gram ===")
for i in terms:
    try:
        top5 = model_sg.wv.most_similar(i, topn=5)
        print(f'{i}: {top5}')
        results_dict['Skip-gram'][i] = top5
    except KeyError:
        print(f'{i}: KeyError')
        results_dict['Skip-gram'][i] = 'KeyError'

# GloVe
print("\n=== GloVe ===")
for i in terms:
    try:
        top5 = glove_model.most_similar(i, topn=5)
        print(f'{i}: {top5}')
        results_dict['GloVe'][i] = top5
    except KeyError:
        print(f'{i}: KeyError')
        results_dict['GloVe'][i] = 'KeyError'



Object IDs:
model id: 2830624487984
model_sg id: 2830625750944
glove_model id: 2830660764480
glove_model type: <class 'gensim.models.keyedvectors.KeyedVectors'>
=== CBOW ===
hamlet: [('good', 0.9929336309432983), ('polo', 0.99224454164505), ('sweet', 0.9905007481575012), ('faith', 0.9880846738815308), ('king', 0.9879695177078247)]
cauldron: [('burned', 0.9985411167144775), ('horse', 0.9984645843505859), ('thereto', 0.9984216094017029), ('opinion', 0.9984073042869568), ('side', 0.9983751177787781)]
nature: [('though', 0.9974195957183838), ('act', 0.9973759055137634), ('even', 0.9972039461135864), ('judgement', 0.9971790313720703), ('whose', 0.9971538186073303)]
spirit: [('wound', 0.9982137680053711), ('hide', 0.9979503154754639), ('lie', 0.9978554844856262), ('bone', 0.9978185296058655), ('vse', 0.9978169798851013)]
general: [('common', 0.9988754391670227), ('danger', 0.9988331198692322), ('seek', 0.9988242387771606), ('draw', 0.9987999200820923), ('vile', 0.9987945556640625)]
prythee: 

Compare the three models by finding the cosine similarity between the following pairs of terms: ('brutus', 'murder'), ('lady macbeth', 'queen gertrude'), ('fortinbras', 'norway'), ('rome', 'norway'), ('ghost', 'spirit'), ('macbeth', 'hamlet'). Comment on how well each model captured the similarity between these terms, especially considering the data that each was trained on.


In [380]:
cosine_tests = [
    ['brutus', 'murder'],
    ['lady macbeth', 'queen gertrude'],
    ['fortinbras', 'norway'],
    ['rome', 'norway'],
    ['ghost', 'hamlet'],
]

for i in cosine_tests:
    w1, w2 = i
    print(f'\nCosine similarity test for: {w1} & {w2}')
    for m in models:
        if id(m) == id(glove_model):
            try:
                print('\n glove model similarity:', m.similarity(w1, w2))
            except KeyError:
                print('KeyError')

        elif id(m) == id(model):
            try:
                print('\n CBOW:', m.wv.similarity(w1, w2))
            except KeyError:
                print('KeyError')
            
        elif id(m) == id(model_sg):
            try:
                print('\n Skip-gram:', m.wv.similarity(w1, w2))
            except KeyError:
                print('KeyError')

            


Cosine similarity test for: brutus & murder
KeyError
KeyError

 glove model similarity: 0.07364358

Cosine similarity test for: lady macbeth & queen gertrude
KeyError
KeyError
KeyError

Cosine similarity test for: fortinbras & norway

 CBOW: 0.9989936

 Skip-gram: 0.94868565

 glove model similarity: -0.028961957

Cosine similarity test for: rome & norway

 CBOW: 0.9955573

 Skip-gram: 0.4303977

 glove model similarity: 0.28583667

Cosine similarity test for: ghost & hamlet

 CBOW: 0.9845966

 Skip-gram: 0.7525937

 glove model similarity: 0.52421004


Compare the three models by finding the 5 most similar terms to each of the following word vectors obtained via linear combination: 'denmark' + 'queen', 'scotland' + 'army' + 'general', 'father' - 'man' + 'woman', 'mother' - 'woman' + 'man'. Comment on how well each model described the ideas behind these word vectors.


In [384]:
analogies = {
    'lc1': {
        "'denmark' + 'queen''": {
            "positive": ['denmark', 'queen'] 
        }
    },
    'lc2': {
        "'scotland' + 'army' + 'general'": {
            'positive': ['scotland', 'army', 'general']
        }
    },
    'lc3': {
        "'father' - 'man' + 'woman'": {
            'positive': ['father', 'woman'],
            'negative': ['man']
        }
    },
    'lc4': {
        "'mother' - 'woman' + 'man'": {
            'positive':['mother', 'man'],
            'negative':['woman']
        }
    }
}


print('\n', '=' * 5, 'GloVe (6B)', '=' * 5)
for lc_id, combos in analogies.items():
    for desc, params in combos.items():
        print(f"\n{desc}:")
        try:
            top5 = glove_model.most_similar(positive=params.get('positive', []), 
                                    negative=params.get('negative', []), 
                                    topn=5)
            for word, score in top5:
                print(f"  {word}: {score:.3f}")
        except KeyError as e:
            print(f"  Missing word: {e}")

print('\n', '=' * 5, 'Skip-gram (Shakespeare)', '=' * 5)
for lc_id, combos in analogies.items():
    for desc, params in combos.items():
        print(f"\n{desc}:")
        try:
            top5 = model_sg.wv.most_similar(positive=params.get('positive', []), 
                                   negative=params.get('negative', []), 
                                    topn=5)
            for word, score in top5:
                print(f"  {word}: {score:.3f}")
        except KeyError as e:
            print(f"  Missing word: {e}")

print('\n', '=' * 5, 'CBOW (Shakespeare)', '=' * 5)
for lc_id, combos in analogies.items():
    for desc, params in combos.items():
        print(f"\n{desc}:")
        try:
            top5 = model.wv.most_similar(positive=params.get('positive', []), 
                                    negative=params.get('negative', []), 
                                    topn=5)
            for word, score in top5:
                print(f"  {word}: {score:.5f}")

        except KeyError as e:
            print(f"  Missing word: {e}")




 ===== GloVe (6B) =====

'denmark' + 'queen'':
  sweden: 0.746
  norway: 0.702
  kingdom: 0.688
  princess: 0.680
  britain: 0.679

'scotland' + 'army' + 'general':
  force: 0.745
  british: 0.734
  military: 0.732
  command: 0.729
  forces: 0.724

'father' - 'man' + 'woman':
  mother: 0.902
  daughter: 0.868
  wife: 0.853
  husband: 0.828
  grandmother: 0.811

'mother' - 'woman' + 'man':
  father: 0.893
  brother: 0.853
  son: 0.822
  uncle: 0.805
  friend: 0.804

 ===== Skip-gram (Shakespeare) =====

'denmark' + 'queen'':
  qu: 0.929
  poison: 0.914
  ophelia: 0.914
  alert: 0.905
  sent: 0.894

'scotland' + 'army' + 'general':
  bowl: 0.974
  freedom: 0.974
  train: 0.970
  descend: 0.970
  lucid: 0.970

'father' - 'man' + 'woman':
  lost: 0.718
  deer: 0.712
  dear: 0.707
  slain: 0.689
  youth: 0.682

'mother' - 'woman' + 'man':
  father: 0.658
  brother: 0.619
  kill: 0.616
  alone: 0.600
  wife: 0.583

 ===== CBOW (Shakespeare) =====

'denmark' + 'queen'':
  alert: 0.99806
  op

Give overall comments on how each model performs. Describe what data you would use to train a better word embedding model to captures the meaning of Shakespearean English.